# Chapter 9: Cost-Sensitive Machine Learning & Threshold Optimization
### *Modern Machine Learning & Data Science Engineering* by Vaibhav Rastogi

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/v893979/Modern_Machine_Learning_and_Data_Science_Engineering/blob/main/src/ch09_hotel_cancellation_cost_sensitive.ipynb)

---

## 📌 Notebook Overview
Learn why accuracy is flawed in asymmetric business problems and optimize decision thresholds on hotel booking cancellations.

In [ ]:
# Environment & Data Auto-Download for Google Colab
import os
import urllib.request

os.makedirs('../data', exist_ok=True)
data_path = "../data/hotel_reservations.csv"
if not os.path.exists(data_path):
    url = "https://raw.githubusercontent.com/v893979/Modern_Machine_Learning_and_Data_Science_Engineering/main/data/hotel_reservations.csv"
    print(f"Downloading hotel_reservations.csv from GitHub...")
    urllib.request.urlretrieve(url, data_path)
    print("Download complete!")


### 1. Defining the Business Loss Matrix

False Negative (\$150 room vacancy loss) vs. False Positive (\$60 overbooking relocation fee):

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/hotel_reservations.csv")
y_true = (df["booking_status"] == "Canceled").astype(int).values
lead_norm = df["lead_time"].values / 365.0

print(f"Total Bookings: {len(df):,} | Cancellation Rate: {y_true.mean():.2%}")

COST_FN = 150.0  # Room sits vacant
COST_FP = 60.0   # Relocation fee for overbooking

### 2. Sweeping Decision Thresholds for Maximum Profit

Evaluate business loss across probability cutoffs from 0.20 to 0.70:

In [ ]:
np.random.seed(42)
y_prob = np.clip(lead_norm * 0.7 + np.random.normal(0, 0.15, size=len(y_true)), 0.01, 0.99)

results = []
for t in [0.20, 0.30, 0.40, 0.50, 0.60, 0.70]:
    y_pred = (y_prob >= t).astype(int)
    fn = np.sum((y_true == 1) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    loss = (fn * COST_FN) + (fp * COST_FP)
    results.append({"Threshold": t, "FN (Vacant)": fn, "FP (Relocated)": fp, "Total Dollar Loss": f"${loss:,.2f}"})

display(pd.DataFrame(results))